# 示例: 4. 马斯京根-康基汇流示例

**脚本:** `examples/run_muskingum_cunge_example.py`

## 目的
演示 `MuskingumCungeRouting` 这一降阶水动力模型的用法。
- **与简单马斯京根法的区别:** 此模块的参数不再是需要率定的 `K` 和 `x`，而是河道的物理参数（长度、坡度、糙率、宽度）。`K` 和 `x` 会在每个时间步根据水流条件动态计算。
- **功能:** 脚本将一个三角洪水波输入模块，并同时输出：
    1.  演算后的出流过程线。
    2.  模拟的河道平均水深过程线。
- **验证:** 最终的图表会同时展示流量演算和水位模拟的结果，验证了该模块的核心功能。

## 1. 环境设置

首先，我们导入所需的库，并把项目的根目录添加到Python的搜索路径中。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))
from hydro_model.routing import MuskingumCungeRouting

## 2. 定义物理参数

与传统的马斯京根法不同，马斯京根-康基法使用物理上可测量的河道参数，而不是经验性的 `K` 和 `x`。这使得模型更具物理意义。

In [ ]:
params = {
    'length': 5000,    # 河段长度 (m)
    'slope': 0.0002,   # 河底坡度 (m/m)
    'manning_n': 0.03, # 曼宁糙率系数
    'width': 50.0,     # 河道宽度 (m)
    'dt': 1.0          # 时间步长 (天)
}

## 3. 创建汇流模块实例

我们使用上面定义的物理参数来创建一个`MuskingumCungeRouting`的实例。

In [ ]:
mc_router = MuskingumCungeRouting(**params)

## 4. 创建入流过程线

我们创建一个与之前示例相同的三角形状的入流过程线，以便进行比较。

In [ ]:
inflow_hydrograph = np.array([0, 10, 20, 40, 60, 45, 30, 20, 10, 5, 2, 1, 0, 0, 0], dtype=float)
timesteps = np.arange(len(inflow_hydrograph))

print(f"输入入流过程线 (m3/s): {inflow_hydrograph}")

## 5. 运行汇流模拟

我们逐个时间步地将入流值输入到汇流模块中。除了收集出流值外，我们还从模块内部获取每个时间步计算出的平均水深。

In [ ]:
outflow_hydrograph = []
water_depth_series = []
for inflow_val in inflow_hydrograph:
    outflow_val = mc_router.run(inflow_val)
    outflow_hydrograph.append(outflow_val)
    water_depth_series.append(mc_router.y_prev)

outflow_hydrograph = np.array(outflow_hydrograph)
water_depth_series = np.array(water_depth_series)

print(f"输出出流过程线 (m3/s): {np.round(outflow_hydrograph, 2)}")
print(f"模拟的平均水深 (m): {np.round(water_depth_series, 2)}")

## 6. 可视化结果

我们创建两个子图来分别展示流量演算和水深模拟的结果。
1.  **流量演算图**: 和马斯京根法类似，洪峰被削减和延迟。
2.  **水深模拟图**: 展示了河道内平均水深随流量的变化过程，这是马斯京-康基法相比于传统马斯京根法一个重要的附加功能。

In [ ]:
plt.figure(figsize=(12, 10))

# 图1: 流量演算
ax1 = plt.subplot(2, 1, 1)
ax1.plot(timesteps, inflow_hydrograph, 'b-o', label='Inflow')
ax1.plot(timesteps, outflow_hydrograph, 'r--s', label='Outflow (Muskingum-Cunge)')
ax1.set_title('Flow Routing Comparison')
ax1.set_xlabel('Time Step (days)')
ax1.set_ylabel('Flow (m³/s)')
ax1.legend()
ax1.grid(True)

# 图2: 水位模拟
ax2 = plt.subplot(2, 1, 2)
ax2.plot(timesteps, water_depth_series, 'g-^', label='Average Water Depth')
ax2.set_title('Simulated In-stream Water Depth')
ax2.set_xlabel('Time Step (days)')
ax2.set_ylabel('Water Depth (m)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()

output_dir = '../examples/results/'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

output_path = os.path.join(output_dir, 'muskingum_cunge_example_plot.png')
plt.savefig(output_path)
print(f"\n演算和水位图已保存到 {output_path}")
plt.show()